In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

we are coding the MLA variant without RoPe first.

In [4]:
class basicMLA(nn.Module):
  def __init__(self, d_model, n_heads, kv_latent_dim):
    super().__init__()
    self.d_model = d_model
    self.n_heads = n_heads
    self.dh = d_model // n_heads  #dim per head

    #projection layers
    self.W_q = nn.Linear(d_model, d_model, bias = False)          #Query projection
    self.W_dkv = nn.Linear(d_model, kv_latent_dim, bias = False)  #compress into latent kv space
    self.W_uk = nn.Linear(kv_latent_dim, d_model , bias = False)  #Decompress k
    self.W_uv = nn.Linear(kv_latent_dim, d_model , bias = False)  #Decompress V
    self.W_o = nn.linear(d_model, d_model, bias = False)          #Final output projection

    self.ln = nn.LayerNorm(kv_latent_dim)
    self.register_buffer('absorbed_k', None)  #Holds W_q @ W_uk

  def forward(self, x, kv_cache=None, past_length = 0):
    B, S, D = x.size()
    # compute absorbed_k once: W_q @ W_uk, shape: (D, latent_dim)
    if self.absorbed_k is None:
      absorbed = torch.matmul(self.W_q.weight, self.W_uk.weight)  #(D, latent_dim)
      self.absorbed_k = absorbed.view(self.n_heads, self.dh, -1)  #(n_heads, dh, latent_dim)

    #compress x into latent kv space
    new_c_kv = self.ln(self.W_dkv(x)) #(B, S, latent_dim)
    if kv_cache is None:
      c_kv = new_c_kv
    else:
      c_kv = torch.cat([kv_cache, new_c_kv], dim = 1)   #(B, S_total, latent_dim)

    S_full = c_kv.size(1)

    # Decompress V to full d_model and split into heads
    v_full = self.W_uv(c_kv) #(B,S_full, D)
    v = v_full.view(B, S_full, self.n_heads, self.dh) #(B, S, n_heads, dh)

    #use input x directly (since W_q is absorbed)
    q = x.view(B, S, self.n_heads, self.dh) #(B, S, n_heads, dh)

    #compute attention scores
    attn_scores = torch.zeros(B, self.n_heads, S, S_full, device = x.device)
    for h in range(self.n_heads):
      tmp = torch.mamtul(q[:,:,h], self.absorbed_k[h])  # (B, S, latent_dim)
      attn_scores[:, h] = torch.bmm(tmp, c_kv.transpose(1,2)) #(B, S, S_full)

    # scale and apply causal mask
    attn_scores = attn_scores / (self.dh ** 0.5)
    mask = torch.tril(torch.ones((S, S_full), device = x.device), diagonal = past_length)
    attn_scores = attn_scores.masked_fill(mask.view(1, 1, S, S_full)==0, float('-inf'))

    # Softmax to get attention weights
    attn_weights = F.softmax(attn_scores, dim = -1) #(B, n_heads, S, S_full)

    # Apply attention weights to each head's V seperately
    out_heads = []
    for h in range(self.n_heads):
      context_h = torch.matmul(attn_weights[:,h], v[:, h]) #(B, S, dh)
      out_heads.append(context_h)
